# Seismic Refraction Tomography

**Near-Surface Geophysics — Missouri S&T**

The intercept-time method treats the ground as a stack of flat layers and asks
the travel-time graph for a handful of numbers: a velocity per layer and a
depth per interface. **Refraction tomography** asks for a picture instead. The
ground becomes a grid of velocities, every first-break pick becomes a ray
through that grid, and the velocities are adjusted until the travel times of
all the rays match all the picks at once.

That needs a lot more data. A velocity at one spot in the ground is only
constrained if rays cross it from several directions, so tomography wants a
shot every one or two geophones, not five shots per line. The worked example
here has sixteen shots into 24 geophones, one at every other geophone plus
off-end shots at both ends.

1. **Read** the records and check the geometry.
2. **Pick** first breaks, bounded by the direct wave.
3. **Check the picks without a model**: reciprocal times.
4. **Fit layers** by the intercept-time method, as a reference.
5. **Understand the forward problem**: rays and travel times through a grid.
6. **Invert**, and let the misfit find the bad picks.
7. **Look at the image** — and at where it is not constrained.
8. **Choose the smoothing**, and see what it costs.
9. **Test what the survey can resolve** with synthetic data.

Units are **metres and seconds**; plots are in milliseconds.

> **The one thing to take away.** A tomogram always looks like geology. The
> colours below the deepest ray are the smoothness constraint, not the
> ground, and a sharp boundary always comes back blurred. Every image below
> is drawn faded where no ray went; read it that way.

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or out of
date) and imports what the notebook uses.

In [ ]:
# Checks for the *tomography* code specifically: an older install imports
# fine and then lacks it, and since the version string does not change
# between builds, a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics[seismic] @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.refraction import refine_picks  # noqa: F401
        from shallowgeo.datasets import example_data  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.refraction.tomography is still missing after installing. "
        "In Colab: Runtime > Restart session, then run this cell again."
    )

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shallowgeo.refraction import (
    PickingSession, load_shots, plot_traveltimes, fit_layers,
    traveltime_tomography, refine_picks, reciprocity, lambda_sweep,
    make_grid, synthetic_traveltimes,
)
from shallowgeo.refraction.tomography import forward
from shallowgeo.datasets import example_data, find_shot_files

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
print("ready")

## 1. The data

By default this uses the **example line that ships with the repository**:
`examples/data/2026-09-25-srt-line/`, sixteen shots saved together in one
SEG-Y file, `3000.sgy`. Its `dataset.md` lists every shot and every known
problem — read it.

To use your own line instead, set `DATA` in the second cell to a folder of
`.sgy` / `.dat` files **or a `.zip` of that folder** — no need to unzip. In
Colab, upload the zip with the file browser on the left (or mount Drive) and
give its path.

In [ ]:
# Found in place in a clone of the repository; anywhere else (Colab) it is
# downloaded once, a few MB, and cached.
DATA = example_data("2026-09-25-srt-line")
print("data from", DATA)

In [ ]:
# --- Your own line instead ---------------------------------------------
# DATA = Path("/content/my-line.zip")        # a .zip, or a folder of .sgy / .dat
# EXCLUDE = ["4011"]                          # files to leave out, by name

EXCLUDE = []
files_found = find_shot_files(DATA)
shots = load_shots(files_found, exclude=EXCLUDE)
print(f"{len(shots)} shot gathers from {len(files_found)} file(s)")

def source_x(survey):
    return float(survey.geometry.table.set_index(["role", "id"]).loc[
        ("source", survey.trace_map["source_id"].iloc[0]), "x"])

inventory = pd.DataFrame([{
    "shot": label,
    "source_x_m": source_x(s),
    "traces": s.n_traces,
    "dt_ms": 1e3 * s.sample_interval,
    "record_ms": 1e3 * s.duration,
    "peak_amplitude": float(np.abs(s.data).max()),
} for label, s in shots.items()]).sort_values("source_x_m").reset_index(drop=True)
display(inventory.round(3))

A single file can hold several field records, and `load_shots` splits them —
here one `.sgy` became sixteen gathers. **Check the geometry before anything
else.** The drawing below should show the geophones and a shot at every
other one, with one or more shots beyond each end.

A second check is worth a minute on any line: the geophone with the largest
early amplitude on each record should be the one nearest the source. If the
header says a shot was at one end and the loudest trace is at the other, the
geometry is wrong and every offset with it. (That happened on the same day
as this line; see "Not included" in `dataset.md`.)

In [ ]:
first = next(iter(shots.values()))
rec_x = first.geometry.receivers["x"].to_numpy(float)
spacing = float(np.diff(np.sort(rec_x)).min())

fig, (ax, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                              gridspec_kw={"height_ratios": [1, 1.4]})
ax.plot(rec_x, np.zeros_like(rec_x), "v", ms=8, color="#256abf", label="geophones")
for k, row in inventory.iterrows():
    ax.plot(row.source_x_m, 0.4 + 0.25 * (k % 2), "*", ms=12, color="#eb6834")
ax.plot([], [], "*", ms=12, color="#eb6834", label="shots")
ax.set_ylim(-0.4, 1.2); ax.set_yticks([]); ax.legend(loc="upper right", fontsize=8)
ax.set_title(f"{len(rec_x)} geophones at {spacing:.3f} m, {len(shots)} shots")

# Loudest trace in the first 15 ms of each record against where the header
# puts the source: the points should sit on the 1:1 line (or at the end
# geophone, for an off-end shot).
for label, s in shots.items():
    n = int(0.015 / s.sample_interval)
    loud = rec_x[np.argmax(np.sqrt((s.data[:, :n] ** 2).mean(axis=1)))]
    ax2.plot(source_x(s), loud, "o", color="#256abf")
lim = [min(rec_x.min(), inventory.source_x_m.min()) - 1,
       max(rec_x.max(), inventory.source_x_m.max()) + 1]
ax2.plot(lim, lim, "k:", lw=1, label="1:1")
ax2.set_xlabel("source position from the header (m)")
ax2.set_ylabel("loudest geophone (m)")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## 2. First breaks

The same two-pass picking as in `refraction_field_data.ipynb`: the AIC picker
with no bound, then a ceiling from the direct wave — no first arrival can be
later than $x/v_0$, so $t_{\max} = 1.5\,x_{\max}/v_0$ throws out picks on
later arrivals before they do any harm.

In [ ]:
session = PickingSession(shots, method="aic", min_time=0.001, min_quality=10.0)
first_pass = session.table()
near = first_pass.nsmallest(max(5, len(first_pass) // 3), "offset")
v0_estimate = float((near["offset"] / near["time"]).median())
MAX_TIME = 1.5 * float(first_pass["offset"].max()) / v0_estimate
session.auto_pick(max_time=MAX_TIME)

print(f"direct wave about {v0_estimate:.0f} m/s -> picks bounded at "
      f"{1e3 * MAX_TIME:.0f} ms")
print(session)
display(session.summary().round(2))

In [ ]:
# Every record with its picks. Red are kept; grey crosses are below the
# quality threshold. Look for picks that jump by several milliseconds from
# one geophone to the next -- the ground cannot change that fast.
labels = session.labels
ncols = 4
nrows = -(-len(labels) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.8 * nrows), squeeze=False)
for ax, label in zip(axes.ravel(), labels):
    session.plot_shot(label, ax=ax, tmax=0.045)
    ax.legend().remove()
    ax.title.set_fontsize(8)
for ax in axes.ravel()[len(labels):]:
    ax.axis("off")
fig.tight_layout(); plt.show()

Most records pick cleanly. Two things stand out on the example line, and both
will come back:

- the three **off-end shots at −2.4 m** (records 3013–3015) are weak, and
  beyond about 8 m their picks wander onto later energy;
- a trace here and there on the good records has jumped to a later arrival.

You could fix these by hand now with `session.widget()` (the cell below), and
on your own data you should look. But tomography offers a second, systematic
way to find them, in section 5.

In [ ]:
# Optional: correct picks by hand. Uncomment to open the picking panel.
# %matplotlib widget          # click-to-pick, where the backend supports it
# session.widget(tmax=0.045)

## 3. Checking the picks without a model: reciprocity

A wave takes the same time from A to B as from B to A. On this line every
shot sits on a geophone, so most source–receiver pairs were recorded **both
ways**, and the two picks should be equal. Their difference is a direct
measurement of picking error that needs no model at all.

This sets expectations for everything that follows: **no model can fit the
picks better than the picks agree with themselves.** It is also where the
error level for the inversion comes from.

In [ ]:
table = session.table()
pairs = reciprocity(table)
print(f"{len(pairs)} reciprocal pairs")
print(f"median |dt| = {1e3 * pairs['dt'].abs().median():.2f} ms,  "
      f"90% below {1e3 * pairs['dt'].abs().quantile(0.9):.2f} ms")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(pairs["offset"], 1e3 * pairs["dt"], "o", ms=4, color="#256abf")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("offset (m)"); ax.set_ylabel("t(A→B) − t(B→A)  (ms)")
ax.set_title("reciprocal pairs: most agree to a few samples; the outliers are mispicks")
ax.grid(alpha=0.3); plt.show()

## 4. The layered answer, for reference

Before any tomography, fit the picks with the intercept-time method, as they
stand. It gives the tomogram something to be compared with — and it is a test
of the picks in its own right.

In [ ]:
layered = fit_layers(table["offset"], table["time"], n_layers=2)
display(layered.summary().to_string())

fig, ax = plt.subplots(figsize=(9, 5))
plot_traveltimes(table, ax=ax, model=layered, legend=False)
ax.set_title("all picks against the two-layer fit")
plt.show()

**Is that believable?** A refractor velocity far above about 6000 m/s is not
a rock, and a refractor several metres deep on a 21 m spread is near the
limit of what it can see. When the first fit says either, suspect the picks
before the ground: a handful of late picks at long offset flattens the far
branch, and a flat branch means an absurdly fast layer. Section 5 finds those
picks; the fit is repeated with the cleaned picks in section 6.

## 5. How tomography computes a travel time

The ground is a grid of **nodes**, each with a velocity. To find the
first-arrival time from a source to every node, join each node to its
neighbours — not only the four nearest, but every node within a few steps in
any direction — and give each link a cost: its length times the average
slowness along it. The quickest route through that network, which Dijkstra's
algorithm finds, is the ray (Fermat's principle) and its total cost is the
travel time. This is the **shortest-path method** (Moser, 1991).

The cell below draws the rays from one shot through a simple two-layer model.
Notice that the rays to the far geophones dive to the fast layer, run along
its top, and come back up: the head wave of the layered method falls out of
Fermat's principle without being put in.

In [ ]:
# A textbook ground, not a fit: 500 m/s over 3000 m/s, interface at 2 m.
# A fine grid (8 nodes per geophone spacing) so the picture is sharp, with
# the interface put on a row of nodes.
DEMO_V1, DEMO_V2, DEMO_H = 500.0, 3000.0, 2.0
demo_grid = make_grid(table, depth=6.0, nodes_per_spacing=8)
DEMO_H = float(demo_grid.z[np.argmin(np.abs(demo_grid.z - DEMO_H))])
X, Z = demo_grid.mesh()
demo_v = np.where(Z < DEMO_H, DEMO_V1, DEMO_V2)
demo_src = float(inventory.source_x_m.iloc[0])
demo = forward(demo_grid, demo_v, np.full(rec_x.size, demo_src), rec_x, rays=True)

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax.imshow(demo_v, extent=(demo_grid.x[0], demo_grid.x[-1], demo_grid.z[-1], 0),
          cmap="viridis", aspect="equal")
for ray in demo["rays"]:
    ax.plot(ray[:, 0], ray[:, 1], "w-", lw=0.8)
ax.plot(rec_x, 0 * rec_x, "kv", ms=4); ax.plot(demo_src, 0, "*", color="#eb6834", ms=12)
ax.set_xlabel("position (m)"); ax.set_ylabel("depth (m)")
ax.set_title(f"shortest-path rays: {DEMO_V1:.0f} over {DEMO_V2:.0f} m/s at {DEMO_H:.2f} m")
off = np.abs(rec_x - demo_src)
ax2.plot(off, 1e3 * demo["time"], "o", label="shortest path through the grid")
xx = np.linspace(0, off.max(), 200)
from shallowgeo.refraction import traveltimes
# Slowness is interpolated between nodes, so the grid's interface acts half a
# node spacing above DEMO_H; the formula is evaluated there.
ax2.plot(xx, 1e3 * traveltimes(xx, [DEMO_V1, DEMO_V2], [DEMO_H - demo_grid.dx / 2]),
         "k-", label="layered formula")
ax2.set_xlabel("offset (m)"); ax2.set_ylabel("time (ms)"); ax2.legend(); ax2.grid(alpha=0.3)
fig.tight_layout(); plt.show()

### Inverting

Each travel time is a sum over the nodes a ray passes of (ray length near the
node) × (slowness of the node). Collecting those lengths gives the **Jacobian**
$J$, and a Gauss–Newton step changes the model $m$ (log-slowness at every
node) to reduce

$$\Phi = \Big\| \frac{t_{\text{obs}} - t(m)}{\sigma} \Big\|^2
          + \lambda \,\| R\,m \|^2 .$$

The first term is the misfit, weighted by the pick error $\sigma$. The second
is the **roughness**: $R$ takes differences between neighbouring nodes, so
$\lambda$ trades fitting the data against keeping the model smooth. Without
it the problem has no unique answer — there are more nodes than picks.
The rays are traced again through each new model, and the loop repeats until
the misfit is at the noise level.

The misfit is reported as $\chi^2$, the mean of the squared weighted
residuals. $\chi^2 \approx 1$ means the model fits the picks exactly as well
as their errors say it should. Much above 1: something in the data cannot be
fitted. Much below: the model is fitting the noise.

**Try it on the picks as they stand.**

In [ ]:
raw = traveltime_tomography(table, verbose=True)
print()
print(raw.summary().to_string())
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
raw.plot_fit(axes); fig.tight_layout(); plt.show()

$\chi^2$ stuck well above 1 and residuals of 10 ms and more: **the model
cannot fit a few of the picks, however it bends.** That is the most useful
thing tomography tells you about picks. A mispick on one trace, with its
neighbours picked correctly, would need the ground to change over one
geophone spacing, and the smoothing will not allow it. So the pick stands
out.

`outliers()` lists the picks more than three robust standard deviations from
the rest. Drop them from the session and invert again — they are not
deleted, only marked unused, and you can look at each one with
`session.plot_shot(...)`.

In [ ]:
outliers = raw.outliers()
print(f"{len(outliers)} outliers")
display(outliers[["shot", "channel", "source_x", "receiver_x", "time",
                  "predicted", "residual"]].assign(
    time=lambda d: 1e3 * d.time, predicted=lambda d: 1e3 * d.predicted,
    residual=lambda d: 1e3 * d.residual).round(2).head(15))

session.exclude(outliers)
cleaner = traveltime_tomography(session.table())
print(f"after dropping them: chi2 {cleaner.chi2:.2f}, RMS {1e3 * cleaner.rms:.2f} ms, "
      f"{len(cleaner.table)} picks, {len(cleaner.outliers())} outliers left")

Now the model is good. Use it to go back to the records: **predict the first
break on every trace — including the dropped ones — and re-pick within a few
milliseconds of the prediction.** Traces where the first pass locked onto a
later, stronger arrival are recovered. The prediction only says where to
look; each pick is still the break the picker finds in that window, so a
model that is wrong somewhere shows up as picks that disagree with it there.

`refine_picks(session)` runs this whole loop — drop outliers until none are
left, guided re-pick, drop outliers again — in one call. It is used here so
the notebook stays short; the two cells above are what it does.

In [ ]:
result = refine_picks(session, verbose=True)
table = session.table()
print()
print(result.summary().to_string())
pairs = reciprocity(table)
print(f"\nreciprocal pairs now: {len(pairs)}, median |dt| "
      f"{1e3 * pairs['dt'].abs().median():.2f} ms")

In [ ]:
# The records again, with the refined picks. Compare with section 2: the
# isolated late picks are gone, and the off-end shots keep only the traces
# where a first break can actually be seen.
fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.8 * nrows), squeeze=False)
for ax, label in zip(axes.ravel(), labels):
    session.plot_shot(label, ax=ax, tmax=0.045)
    ax.legend().remove(); ax.title.set_fontsize(8)
for ax in axes.ravel()[len(labels):]:
    ax.axis("off")
fig.tight_layout(); plt.show()

## 6. The velocity image

First, the layered fit again, now on the cleaned picks.

In [ ]:
layered_raw = layered
layered = fit_layers(table["offset"], table["time"], n_layers=2)
print("raw picks:    " + " over ".join(f"{v:.0f}" for v in layered_raw.velocities)
      + f" m/s, interface at {layered_raw.depths[0]:.1f} m")
print("clean picks:  " + " over ".join(f"{v:.0f}" for v in layered.velocities)
      + f" m/s, interface at {layered.depths[0]:.1f} m")

Now the image. Three things are drawn on it:

- **faded** where fewer than one ray's worth of path crosses a node — the
  colours there come from the smoothness constraint, not from data;
- **velocity contours**, the usual way to trace a boundary on a tomogram;
- the **intercept-time interface** from the cleaned picks, dashed.

In [ ]:
CONTOURS = [750, 1500, 2500, 3500]

fig, axes = plt.subplots(2, 1, figsize=(11, 8.5))
result.plot(ax=axes[0], contours=CONTOURS)
for d in np.atleast_1d(layered.depths):
    axes[0].axhline(d, color="#eb6834", ls="--", lw=1.5, label="intercept-time interface")
axes[0].legend(loc="lower right", fontsize=8)
axes[0].set_title(f"velocity — chi² {result.chi2:.2f}, RMS {1e3 * result.rms:.2f} ms, "
                  f"{len(result.table)} picks")
result.plot(ax=axes[1], rays=True, contours=CONTOURS)
axes[1].set_title("the rays through the final model")
fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
result.plot(ax=axes[0], field="coverage")
axes[0].set_title("ray coverage: metres of ray through each node's cell")
result.plot(ax=axes[1], field="change")
axes[1].set_title("change from the starting gradient (%)")
fig.tight_layout(); plt.show()

doi = result.depth_of_investigation()
inside = (result.grid.x >= rec_x.min()) & (result.grid.x <= rec_x.max())
print(f"rays reach {np.nanmin(doi[inside]):.1f} to {np.nanmax(doi):.1f} m below the "
      f"surface under the spread (spread length {np.ptp(rec_x):.1f} m)")

**Velocity–depth profiles** make the comparison with the layered model
precise. The tomogram replaces the step at the interface with a gradient
several metres thick — both images fit the picks, and the data cannot tell
them apart. A common convention is to draw the interface at a velocity
contour between the two layer velocities; where the step sits within the
gradient is what the intercept-time depth adds.

In [ ]:
positions = np.percentile(rec_x, [15, 50, 85])
fig, axes = plt.subplots(1, len(positions), figsize=(13, 5), sharey=True)
zz = np.linspace(0, result.grid.z[-1], 200)
v_layer = np.where(zz < layered.depths[0], layered.velocities[0], layered.velocities[1])
for ax, x in zip(axes, positions):
    prof = result.profile(x)
    good = prof["coverage"] >= result.grid.dx
    ax.plot(prof["velocity"], prof["depth"], "-", color="0.75", lw=2, label="unconstrained")
    ax.plot(prof["velocity"].where(good), prof["depth"], "-", color="#256abf", lw=2.5,
            label="tomography")
    ax.plot(prof["start"], prof["depth"], ":", color="0.4", label="starting model")
    ax.plot(v_layer, zz, "--", color="#eb6834", lw=2, label="two layers")
    ax.set_title(f"x = {x:.1f} m"); ax.set_xlabel("velocity (m/s)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("depth (m)"); axes[0].invert_yaxis(); axes[0].legend(fontsize=8)
fig.tight_layout(); plt.show()

### The residuals

Always look at them. Three panels: predicted against observed, residual
against offset, and residual by shot.

- A residual that **trends with offset** — positive at short offsets and
  negative at long ones, or the reverse — means the model is systematically
  too slow near the surface and too fast at depth, or vice versa. On this
  line that is the smoothing blurring a sharp interface; try a smaller
  $\lambda$ in the next section and watch the trend shrink.
- A **shot whose residuals all sit off zero** has a timing problem — a
  trigger that fired early or late — or a wrong source position.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
result.plot_fit(axes); fig.tight_layout(); plt.show()

## 7. How smooth? Choosing λ

$\lambda$ is the one number you have to choose, and it changes the picture.
The usual rule: take the **smoothest model that still fits the data to the
noise level**, $\chi^2 \approx 1$. Smaller $\lambda$ fits better and grows
structure; below the noise level that structure is fitting pick errors.

`z_weight` is the second knob: the weight of vertical relative to horizontal
smoothing. Below 1 lets velocity change faster with depth than along the
line, which is usually how the ground is.

In [ ]:
LAMS = (3, 10, 30, 100, 300)
sweep = lambda_sweep(table, lams=LAMS)
display(sweep.drop(columns="result").round(3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep["roughness"], sweep["chi2"], "o-", color="#256abf")
for _, row in sweep.iterrows():
    ax.annotate(f"λ={row.lam:g}", (row.roughness, row.chi2),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.axhline(1, color="k", ls=":", lw=1)
ax.set_xlabel("model roughness |R m|"); ax.set_ylabel("chi²")
ax.set_title("fit against roughness"); ax.grid(alpha=0.3); plt.show()

fig, axes = plt.subplots(3, 1, figsize=(10, 10))
for ax, lam in zip(axes, (3, 30, 300)):
    res = sweep.set_index("lam").loc[lam, "result"]
    res.plot(ax=ax, contours=CONTOURS, vmin=250, vmax=5000)
    ax.set_title(f"λ = {lam}: chi² {res.chi2:.2f}, RMS {1e3 * res.rms:.2f} ms")
fig.tight_layout(); plt.show()

## 8. What could this survey have seen? A synthetic test

The only way to know what an inversion can recover is to give it data from a
model you chose. Below: a ground you design, travel times computed through
it for **exactly the same source–receiver pairs as the real picks**, noise
at the reciprocal-time level added, and the same inversion run on them.

Change `true_velocity` and run it again. Things worth trying: a thinner or
thicker top layer; a dipping interface; a low-velocity zone under a fast
layer (it will not appear — first arrivals never visit it); a narrow block
of fast rock in the middle (how narrow before it disappears?).

In [ ]:
syn_grid = result.grid
X, Z = syn_grid.mesh()

# --- design the ground here -----------------------------------------------
interface = 1.5 + 0.08 * (X - X.min())               # a dipping boundary
true_velocity = np.where(Z < interface, 450 + 80 * Z, 3500.0)
# true_velocity[(np.abs(X - 10) < 1.5) & (Z < 1.2)] = 1200   # a buried block?
# ---------------------------------------------------------------------------

noise = float(np.median(np.abs(pairs["dt"]))) / np.sqrt(2)
syn = table[["shot", "source_x", "receiver_x"]].copy()
syn["time"] = forward(syn_grid, true_velocity, syn["source_x"], syn["receiver_x"],
                      jacobian=False)["time"]
syn["time"] += np.random.default_rng(0).normal(0, noise, len(syn))

syn_result = traveltime_tomography(syn, grid=syn_grid, lam=result.settings["lam"],
                                   z_weight=result.settings["z_weight"])
# Same error model as the real inversion, so chi² comes out below 1 when the
# added noise is smaller than the errors that model assumes.
print(f"noise {1e3 * noise:.2f} ms -> chi² {syn_result.chi2:.2f}, "
      f"RMS {1e3 * syn_result.rms:.2f} ms")

fig, axes = plt.subplots(2, 1, figsize=(10, 7.5))
true_res = type(syn_result)(syn_grid, true_velocity, syn_result.start, syn_result.coverage,
                            syn_result.table, syn_result.history, syn_result.settings)
true_res.plot(ax=axes[0], mask=False, vmin=250, vmax=4000, contours=CONTOURS)
axes[0].set_title("the ground you designed")
syn_result.plot(ax=axes[1], vmin=250, vmax=4000, contours=CONTOURS)
axes[1].plot(X[0], interface[0], "--", color="#eb6834", lw=1.5, label="true interface")
axes[1].legend(loc="lower right", fontsize=8)
axes[1].set_title("what the inversion recovers from the same rays")
fig.tight_layout(); plt.show()

## 9. Optional: the same inversion in pyGIMLi

[pyGIMLi](https://www.pygimli.org) is the research code this package will use
for joint inversion. It solves the same problem on an unstructured triangle
mesh with its own ray tracer. Running both on the same picks is a check on
each: where they agree the answer does not depend on the software.

pyGIMLi installs with `pip` on Colab and Linux (about a minute); on your own
machine use the `full` pixi environment. Set `TRY_PYGIMLI = True` to run it.

In [ ]:
TRY_PYGIMLI = False

if TRY_PYGIMLI:
    try:
        import pygimli  # noqa: F401
    except ImportError:
        %pip install -q pygimli
    pg_result = traveltime_tomography(table, grid=result.grid, backend="pygimli",
                                      lam=result.settings["lam"],
                                      z_weight=result.settings["z_weight"])
    print(pg_result.summary().to_string())
    both = result.covered() & pg_result.covered()
    diff = 100 * np.abs(result.velocity - pg_result.velocity)[both] / pg_result.velocity[both]
    print(f"\nwhere both have rays, the velocities differ by a median of "
          f"{np.median(diff):.0f}%")
    fig, axes = plt.subplots(2, 1, figsize=(10, 7.5))
    result.plot(ax=axes[0], contours=CONTOURS, vmin=250, vmax=5000)
    axes[0].set_title("shallowgeo (shortest-path rays, regular grid)")
    pg_result.plot(ax=axes[1], contours=CONTOURS, vmin=250, vmax=5000)
    axes[1].set_title(f"pyGIMLi {pg_result.settings['pygimli_version']} "
                      "(unstructured mesh), resampled to the same grid")
    fig.tight_layout(); plt.show()

## 10. Save and write up

In [ ]:
OUT = Path("tomography_output")
OUT.mkdir(exist_ok=True)
session.save(OUT / "picks.csv")
X, Z = result.grid.mesh()
pd.DataFrame({"x_m": X.ravel(), "depth_m": Z.ravel(),
              "velocity_m_s": result.velocity.ravel(),
              "ray_coverage_m": result.coverage.ravel()}).to_csv(OUT / "velocity.csv",
                                                                  index=False)

s = result.summary()
print("\n".join([
    f"LINE        {len(rec_x)} geophones at {spacing:.2f} m, {np.ptp(rec_x):.1f} m spread",
    f"SHOTS       {len(shots)}, {len(result.table)} picks used of "
    f"{len(session.table(used_only=False))}",
    f"PICK ERROR  reciprocal pairs agree to {1e3 * pairs['dt'].abs().median():.2f} ms (median)",
    f"INVERSION   lam = {s.lam:g}, z_weight = {s.z_weight:g}: chi² {s.chi2:.2f}, "
    f"RMS {s.rms_ms:.2f} ms",
    f"VELOCITY    {s.v_min:.0f} to {s.v_max:.0f} m/s where rays go",
    f"DEPTH       constrained to about {s.max_depth_m:.1f} m",
    f"LAYERED     " + " over ".join(f"{v:.0f}" for v in layered.velocities)
    + f" m/s, interface at {layered.depths[0]:.1f} m",
    "",
    f"written to {OUT.resolve()}",
]))

### Questions

1. Where does the 1500 m/s contour sit relative to the intercept-time
   interface? Why would the two methods disagree about depth even when both
   fit the picks?
2. How deep does the tomogram see? Compare with the rule of thumb of a
   quarter to a third of the spread length.
3. Pick the λ you would report. Defend it with the χ² and the images, not
   with which picture looks best.
4. In the synthetic test, how narrow can a buried block be before the
   inversion no longer shows it? What sets that limit — geophone spacing,
   shot spacing, or depth?
5. The three off-end shots at −2.4 m contribute few picks. What would you
   change in the field to make them useful?
6. The same day's shots at the *other* geophones (files `2`–`15`) were
   recorded two hours earlier and their reciprocal times are about 0.9 ms
   off from this line's. What would you have to do before inverting the two
   together, and what would you gain?

### What to do differently next time

- **Shoot at every geophone** when you can. Every-other is enough for an
  image; every one doubles the crossing rays at no extra cost in geophones.
- **Off-end shots further out, and stacked.** They carry the deepest rays.
  A weak blow on soft ground at the end of the line wastes them.
- **Record the geometry on paper**, especially the direction the shots walk.
  A header that says "0 ft" is only as good as whoever typed it.
- **Survey the elevations.** Tomography can use topography
  (`make_grid(..., elevation=...)`); a metre of relief ignored on a 20 m
  line is a velocity error, not a detail.